# Customer Transaction Segmentation with RFM

Minimal implementation: definitions (with comments), training, prediction.

In [ ]:
# ---------------------------------------------------------------------------
# Shared numeric/pandas imports and one date helper
# ---------------------------------------------------------------------------
import numpy as np  # numeric cleanup, infinities, probability arrays
import pandas as pd  # CSV loading, cleaning, feature engineering


def _dates(values):
    # Parse ISO dates first, then day-first dates; unreadable text becomes NaT.
    s = values.astype('string')  # string dtype keeps missing values as <NA>
    parsed = pd.to_datetime(s, errors='coerce', format='%Y-%m-%d %H:%M')
    missing = parsed.isna()  # ISO format could not read these rows
    if missing.any():
        parsed.loc[missing] = pd.to_datetime(s[missing], errors='coerce', dayfirst=True)
    return parsed


# Only the imports this task actually uses.
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# ---------------------------------------------------------------------------
# RFM contract: three features per customer, no target labels
# ---------------------------------------------------------------------------
RFM_COLUMNS = ['recency_days', 'frequency', 'monetary']  # exact model input order
REFERENCE_DATE = pd.Timestamp('2026-12-01')  # fixed "today" for recency calculations


def load_data(filename):
    # A1: collapse invoice lines into one RFM row per valid customer.
    df = pd.read_csv(filename)
    # IDs arrive as "1000.0"; make them clean strings and remove blanks.
    df['customer_id'] = (df['customer_id'].astype('string').str.strip()
                         .str.replace(r'\.0$', '', regex=True).fillna(''))
    df['invoice_no'] = df['invoice_no'].astype('string').str.strip().fillna('')
    df['invoice_date'] = _dates(df['invoice_date'])
    for c in ('quantity', 'unit_price'):
        df[c] = pd.to_numeric(df[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
    # Drop bad keys, cancellations (invoice starts with C), non-positive amounts and future dates.
    df = df[(df['customer_id'] != '') & (df['invoice_no'] != '') & df['invoice_date'].notna()].copy()
    df = df[~df['invoice_no'].str.upper().str.startswith('C')]
    df = df[(df['quantity'] > 0) & (df['unit_price'] > 0) & (df['invoice_date'] < REFERENCE_DATE)].copy()
    df['amount'] = df['quantity'] * df['unit_price']  # line total
    # Group by customer, sort=False keeps first-appearance order from the CSV.
    groups = df.groupby('customer_id', sort=False).agg(
        last_purchase=('invoice_date', 'max'),
        frequency=('invoice_no', 'nunique'),  # distinct invoices, not line count
        monetary=('amount', 'sum'),
    ).reset_index()
    # Normalize to midnight so a purchase on 25 Nov is exactly 6 days ago.
    groups['recency_days'] = (REFERENCE_DATE - groups['last_purchase'].dt.normalize()).dt.days
    groups['frequency'] = groups['frequency'].astype(int)
    return groups[['customer_id'] + RFM_COLUMNS]  # ID first, then the three features


def build_pipeline(n_clusters):
    # A2: numeric-only clustering pipeline; log1p is valid because RFM >= 0.
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
        ('scaler', StandardScaler()),  # KMeans uses distances, so scaling matters
        ('model', KMeans(n_clusters=n_clusters, random_state=42, n_init=10)),
    ])


def prepare_datasets(train_file, test_file):
    # 1: no labels exist here; just return cleaned customer tables.
    return load_data(train_file), load_data(test_file)


def train_models(train_customers):
    # 2: fit three candidate cluster counts on the same three RFM features.
    models = {f'kmeans_{k}': build_pipeline(k) for k in (3, 4, 5)}
    for name, model in models.items():
        models[name] = model.fit(train_customers[RFM_COLUMNS])
    return models


def evaluate_models(trained_models, test_customers):
    # 3: score on held-back customers in the transformed feature space.
    X = test_customers[RFM_COLUMNS]
    rows = []
    for name, model in trained_models.items():
        labels = model.predict(X)  # cluster assignment for each customer
        transformed = model[:-1].transform(X)  # same space KMeans was fitted in
        valid = 1 < len(np.unique(labels)) < len(X)  # metrics need 2..n-1 clusters
        rows.append({
            'model_name': name,
            'silhouette_score': silhouette_score(transformed, labels) if valid else -1.0,
            'davies_bouldin_score': davies_bouldin_score(transformed, labels) if valid else np.nan,
        })
    return (pd.DataFrame(rows)
            .sort_values(['silhouette_score', 'davies_bouldin_score', 'model_name'],
                         ascending=[False, True, True], na_position='last')
            .reset_index(drop=True))  # higher silhouette and lower DB are better


def select_best_model(comparison, trained_models):
    # 4: take the already-ranked winner from the dictionary, no retraining.
    name = comparison.iloc[0]['model_name']
    return name, trained_models[name]


def segment_customers(model, customers):
    # 5: assign segment IDs to new customers, preserving their order.
    return pd.DataFrame({
        'customer_id': customers['customer_id'].to_numpy(),
        'segment_id': model.predict(customers[RFM_COLUMNS]).astype(int),  # KMeans labels are integers
    })


def train():
    # TRAINING WORKFLOW: build RFM tables, fit candidates, save winner + report.
    train_customers, test_customers = prepare_datasets('retail_train.csv', 'retail_test.csv')
    models = train_models(train_customers)
    comparison = evaluate_models(models, test_customers)
    name, best = select_best_model(comparison, models)
    joblib.dump(best, 'best_model.pkl')
    comparison.to_csv('model_comparison.csv', index=False)
    return name, comparison


def predict():
    # PREDICTION WORKFLOW: reload saved pipeline, segment new customers only.
    model = joblib.load('best_model.pkl')
    result = segment_customers(model, load_data('retail_predict.csv'))
    result.to_csv('retail_segments.csv', index=False)
    return result


In [ ]:
# TRAINING WORKFLOW: this is the only cell that fits models and writes artifacts.
name, comparison = train()
comparison

In [ ]:
# PREDICTION WORKFLOW: reload the saved model; no fitting happens here.
predict()